# 03 — Docker and Azure Deployment Walkthrough

This notebook is deliberately **markdown-heavy**, not code-heavy. Its subject — building a Docker
image and deploying it to Azure — is fundamentally about shell commands and CLI tooling (`docker`,
`az`), not something meaningfully "run" inside a notebook kernel. Actually invoking Docker or the Azure
CLI here would require a Docker daemon and real Azure credentials, which breaks this course's
"runs offline, no external services" rule for notebooks.

Instead, this notebook walks through the commands **as annotated reference material** — read it
side-by-side with chapters 03 (Docker) and 04 (Azure App Service / Functions) — and the few code cells
it does contain are safe, introspective Python (e.g., printing the Dockerfile as a string, so you can
see exactly what would be built) rather than anything that touches Docker or Azure.


## 1. The Dockerfile for the service from notebook 1

This is the same multi-stage Dockerfile discussed in chapter 03, written for the FastAPI CRUD app built
in `01_fastapi_crud_api_demo.ipynb`. The next code cell holds it as a Python string purely so it can be
displayed and inspected inline — it is not executed, built, or run against a Docker daemon anywhere in
this notebook.


In [1]:
dockerfile_contents = r"""
# ---- Stage 1: install dependencies ----
FROM python:3.11-slim AS builder
WORKDIR /build
COPY requirements.txt .
RUN pip install --user --no-cache-dir -r requirements.txt

# ---- Stage 2: slim runtime image ----
FROM python:3.11-slim
WORKDIR /app

COPY --from=builder /root/.local /root/.local
ENV PATH=/root/.local/bin:$PATH

COPY . .

RUN useradd --create-home appuser
USER appuser

EXPOSE 8000

# uvicorn (an ASGI server) running the FastAPI app object directly.
# For a multi-worker production process manager instead, swap this for:
#   CMD ["gunicorn", "app.main:app", "-k", "uvicorn.workers.UvicornWorker", "--workers", "4", "--bind", "0.0.0.0:8000"]
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

print(dockerfile_contents)
print(f"Dockerfile is {len(dockerfile_contents.splitlines())} lines.")



# ---- Stage 1: install dependencies ----
FROM python:3.11-slim AS builder
WORKDIR /build
COPY requirements.txt .
RUN pip install --user --no-cache-dir -r requirements.txt

# ---- Stage 2: slim runtime image ----
FROM python:3.11-slim
WORKDIR /app

COPY --from=builder /root/.local /root/.local
ENV PATH=/root/.local/bin:$PATH

COPY . .

RUN useradd --create-home appuser
USER appuser

EXPOSE 8000

# uvicorn (an ASGI server) running the FastAPI app object directly.
# For a multi-worker production process manager instead, swap this for:
#   CMD ["gunicorn", "app.main:app", "-k", "uvicorn.workers.UvicornWorker", "--workers", "4", "--bind", "0.0.0.0:8000"]
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]

Dockerfile is 25 lines.


## 2. Building the image locally

Run these from a terminal, in the directory containing the Dockerfile and the FastAPI app code (the
`app.main:app` object built from chapters 02/06, plus a `requirements.txt` listing `fastapi`, `uvicorn`,
`sqlalchemy`, `msal`, `pyjwt`, and the rest of the dependency set from chapter 03).

```bash
# Build the image, tagging it with a specific version (never rely only on "latest" — chapter 03)
docker build -t doc-uploader:1.0.0 .

# Confirm the image exists locally and check its size
docker images doc-uploader
```

**What to expect:** the first build will take longer (installing dependencies from scratch). Re-running
`docker build` after only editing application code (not `requirements.txt`) should be noticeably
faster — that's the layer-caching behavior from chapter 03: the `pip install` layer is reused because
its only input (`requirements.txt`) hasn't changed.


## 3. Running the container locally

```bash
# Run the container, mapping the container's port 8000 to the host's port 8000
docker run --rm -p 8000:8000 --name doc-uploader-local doc-uploader:1.0.0

# In a second terminal, exercise it exactly like the notebook 1 TestClient calls, but over real HTTP.
# A real request also needs a bearer token (chapter 06) — this curl uses the same offline-style fake
# token shape notebook 1 uses, which only works if the deployed app is running with the mocked auth
# dependency from that demo; the real service validates a genuine Azure AD-issued token instead.
curl -X POST http://localhost:8000/v1/documents \
  -H "Authorization: Bearer role:DocumentUploader.Write;user:alice@hsbc.com" \
  -F "file=@sample.pdf"

curl http://localhost:8000/v1/documents \
  -H "Authorization: Bearer role:DocumentUploader.Read;user:bob@hsbc.com"
```

**Why this matters for the interview:** notebook 1 proved the route logic is correct using FastAPI's
in-process `TestClient`. This step proves the *containerized* app — running under `uvicorn` (or
`gunicorn` with `uvicorn.workers.UvicornWorker`), inside the image built above — behaves the same way
over a real HTTP socket. Both checks matter: the test client catches logic bugs fast in CI; the
container run catches packaging/environment bugs (missing dependency, wrong `WEBSITES_PORT`, wrong
entrypoint) that the test client can't see.


## 4. Pushing to Azure Container Registry (ACR)

```bash
# Authenticate to the registry (uses your `az login` session or a service principal)
az acr login --name myregistry

# Tag the local image for the registry
docker tag doc-uploader:1.0.0 myregistry.azurecr.io/doc-uploader:1.0.0

# Push
docker push myregistry.azurecr.io/doc-uploader:1.0.0
```

Tagging with a specific semantic version (`1.0.0`), not just `latest`, is what makes a later rollback
(`az webapp config container set ... --container-image-name myregistry.azurecr.io/doc-uploader:1.3.9`)
possible — chapter 03 covers why relying solely on `latest` makes "what's actually running in prod"
unanswerable.


## 5. Deploying to Azure App Service (Web App for Containers)

Chapter 04's "always-on, request/response" workload — the FastAPI service itself.

```bash
# One-time: create the Web App, pointing it at the pushed container image
az webapp create \
  --name doc-uploader-svc \
  --resource-group my-rg \
  --plan my-app-service-plan \
  --deployment-container-image-name myregistry.azurecr.io/doc-uploader:1.0.0

# Tell App Service which port the container listens on internally (matches uvicorn's --port 8000)
az webapp config appsettings set \
  --name doc-uploader-svc \
  --resource-group my-rg \
  --settings WEBSITES_PORT=8000

# Subsequent deploys: point the app at a new image tag (this is what a CD pipeline automates)
az webapp config container set \
  --name doc-uploader-svc \
  --resource-group my-rg \
  --container-image-name myregistry.azurecr.io/doc-uploader:1.1.0
```

**Deployment slots** (chapter 04) — deploy to a staging slot first, verify, then swap with zero
downtime:

```bash
az webapp deployment slot create --name doc-uploader-svc --resource-group my-rg --slot staging

az webapp config container set \
  --name doc-uploader-svc --resource-group my-rg --slot staging \
  --container-image-name myregistry.azurecr.io/doc-uploader:1.1.0

# After smoke-testing the staging slot's own URL:
az webapp deployment slot swap --name doc-uploader-svc --resource-group my-rg \
  --slot staging --target-slot production
```


## 6. Deploying the post-upload processor as an Azure Function

Chapter 04's "event-driven, bursty" workload — reacting to a new blob, not sitting always-on.

```bash
# One-time: create the Function App (consumption plan = pay-per-execution, scales to zero)
az functionapp create \
  --name doc-uploader-processor \
  --resource-group my-rg \
  --storage-account myfuncstorage \
  --consumption-plan-location eastus \
  --runtime python \
  --runtime-version 3.11 \
  --functions-version 4

# Deploy the function code (blob-triggered function from chapter 04)
func azure functionapp publish doc-uploader-processor
```

The trigger binding itself (`path="uploads/{name}"`, `connection="AzureWebJobsStorage"`) is declared in
the function's code/config, not on the CLI — see chapter 04's `process_uploaded_document` example for
the full trigger definition.


## 7. Sanity-checking the deployment

```bash
# Tail App Service logs
az webapp log tail --name doc-uploader-svc --resource-group my-rg

# Check the Function's recent invocation history
az functionapp show --name doc-uploader-processor --resource-group my-rg --query state

# Hit the deployed API directly (still needs a real Azure AD-issued bearer token in production, chapter 06)
curl https://doc-uploader-svc.azurewebsites.net/v1/documents \
  -H "Authorization: Bearer <real-azure-ad-token>"
```

If `WEBSITES_PORT` doesn't match the port `uvicorn` binds to inside the container (`8000` in this
Dockerfile), the most common symptom is App Service returning `502 Bad Gateway` even though the
container builds and starts successfully — worth remembering as a debugging checklist item, since it's
a very easy setting to forget or mistype.


## Summary

| Step | Command family | Chapter reference |
|---|---|---|
| Build image | `docker build` | 03 |
| Run locally | `docker run` | 03 |
| Push to registry | `az acr login`, `docker push` | 03 |
| Deploy REST API | `az webapp create` / `config container set` | 04 |
| Zero-downtime release | `az webapp deployment slot ...` | 04 |
| Deploy async processor | `az functionapp create`, `func azure functionapp publish` | 04 |
| Verify | `az webapp log tail`, `curl` (with a bearer token, chapter 06) | 04, 06 |

None of these commands were executed by this notebook — they're reference material to run from an
actual terminal with Docker Desktop and an authenticated `az` CLI session. The only code that ran in
this notebook was the safe, introspective Dockerfile-printing cell in section 1.
